In [ ]:
## This is Just for my Project work and every data files i accessed and used are publicly available on ECI and Kaggle and dont take this serious cause these are my assumptions here
## ECI produced data are scanned images - i worked on tesseract + OCRimage + pdf2image to extract data into excel using Python (VScode) and cleaned the dataset using excel and python 
## All these codes written using Help of Claude AI.
## Author - Prasanth Sundar (P S) LinkedIN--(https://www.linkedin.com/in/prasanth-sundar-b65475359/) Github--(https://github.com/Prashantsundar)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from collections import Counter
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (13, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
np.random.seed(2024)

FILE_2021 = r(copy file path)
FILE_2026 = r(copy file path)

print('✅ Cell 1 complete — imports and paths loaded')

In [ ]:


raw2021 = pd.read_csv(FILE_2021)
elec2026_raw = pd.read_csv(FILE_2026)

print(f'✅ 2021 file loaded  → {len(raw2021):,} rows | {raw2021["Constituency"].nunique()} constituencies')
print(f'✅ 2026 file loaded  → {len(elec2026_raw):,} rows')
print()
print('2021 columns:', raw2021.columns.tolist())
print('2026 columns:', elec2026_raw.columns.tolist())
print()
print('--- 2021 sample ---')
display(raw2021.head(3))
print('--- 2026 sample ---')
display(elec2026_raw.head(3))

In [ ]:

# CELL 3 — PARTY MAPPING & BUILD AC-LEVEL PIVOT


# Map long party names → short codes
PARTY_SHORT = {
    'Dravida Munnetra Kazhagam':                        'DMK',
    'All India Anna Dravida Munnetra Kazhagam':         'AIADMK',
    'Indian National Congress':                         'INC',
    'Bharatiya Janata Party':                           'BJP',
    'Naam Tamilar Katchi':                              'NTK',
    'Pattali Makkal Katchi':                            'PMK',
    'Viduthalai Chiruthaigal Katchi':                   'VCK',
    'Communist Party of India  (Marxist)':              'CPI-M',
    'Communist Party of India':                         'CPI',
    'Marumalarchi Dravida Munnetra Kazhagam':           'MDMK',
    'Indian Union Muslim League':                       'IUML',
    'Desiya Murpokku Dravida Kazhagam':                 'DMDK',
    'Makkal Needhi Maiam':                              'MNM',
    'Amma Makkal Munnettra Kazagam':                    'AMMK',
    'None of the Above':                                'NOTA',
}

DMK_ALLIANCE_PARTIES    = {'DMK','INC','VCK','CPI-M','CPI','MDMK','IUML'}
AIADMK_ALLIANCE_PARTIES = {'AIADMK','PMK','BJP','DMDK'}

raw2021['Party_Short'] = raw2021['Party'].map(PARTY_SHORT).fillna('Others')

def get_alliance(p):
    if p in DMK_ALLIANCE_PARTIES:    return 'DMK_Alliance'
    if p in AIADMK_ALLIANCE_PARTIES: return 'AIADMK_Alliance'
    return 'Others'

raw2021['Alliance'] = raw2021['Party_Short'].apply(get_alliance)

# ── AC-level vote counts per party ────────────────────────────────────────────
ac_party = (
    raw2021
    .groupby(['Constituency','Party_Short'])['Total_Votes']
    .sum().reset_index()
)
ac_pivot = ac_party.pivot_table(
    index='Constituency', columns='Party_Short',
    values='Total_Votes', fill_value=0
).reset_index()

# ── Per-AC total votes & winner ────────────────────────────────────────────────
ac_meta = raw2021.groupby('Constituency').agg(
    Total_Polled=('Tot_Constituency_votes_polled','first')
).reset_index()

winners_2021 = (
    raw2021[raw2021['Win_Lost_Flag']=='WIN']
    [['Constituency','Party_Short','Alliance','Total_Votes','Winning_votes']]
    .copy()
    .rename(columns={
        'Party_Short':'Winner_Party',
        'Alliance':'Winner_Alliance',
        'Total_Votes':'Winner_Votes'
    })
)

# ── Merge everything into ac_df ────────────────────────────────────────────────
ac_df = ac_pivot.merge(ac_meta, on='Constituency').merge(winners_2021, on='Constituency')

# ── Alliance-level vote shares ─────────────────────────────────────────────────
dmk_cols    = [c for c in ac_df.columns if c in ['DMK','INC','VCK','CPI-M','CPI','MDMK','IUML']]
aiadmk_cols = [c for c in ac_df.columns if c in ['AIADMK','PMK','BJP','DMDK']]

ac_df['DMK_Alliance_Votes']    = ac_df[dmk_cols].sum(axis=1)
ac_df['AIADMK_Alliance_Votes'] = ac_df[aiadmk_cols].sum(axis=1)
ac_df['DMK_Alliance_Share']    = ac_df['DMK_Alliance_Votes']    / ac_df['Total_Polled']
ac_df['AIADMK_Alliance_Share'] = ac_df['AIADMK_Alliance_Votes'] / ac_df['Total_Polled']

ntk_votes = ac_df['NTK'] if 'NTK' in ac_df.columns else 0
ac_df['NTK_share']    = ntk_votes / ac_df['Total_Polled']
ac_df['Others_share'] = (1 - ac_df['DMK_Alliance_Share']
                           - ac_df['AIADMK_Alliance_Share']
                           - ac_df['NTK_share']).clip(0)
ac_df['DMK_Won']      = (ac_df['Winner_Alliance'] == 'DMK_Alliance').astype(int)

print(f'✅ Cell 3 complete — ac_df built with {len(ac_df)} rows')
print(f"   DMK Alliance wins  : {ac_df['DMK_Won'].sum()}")
print(f"   AIADMK Alliance    : {(ac_df['Winner_Alliance']=='AIADMK_Alliance').sum()}")
print(f"   Others             : {(ac_df['Winner_Alliance']=='Others').sum()}")
ac_df[['Constituency','Winner_Party','Winner_Alliance',
       'DMK_Alliance_Share','AIADMK_Alliance_Share','NTK_share']].head(5)

In [ ]:

# CELL 4 — CLEAN & MERGE 2026 ELECTOR ROLLS


elec2026 = elec2026_raw.copy()

# Rename columns (handles both possible column name formats)
col_rename = {}
for c in elec2026.columns:
    cl = c.strip()
    if 'Name' in cl and 'Assembly' in cl:  col_rename[c] = 'Constituency'
    elif cl == 'Total':                     col_rename[c] = 'Electors_2026'
    elif cl == 'Male':                      col_rename[c] = 'Male_2026'
    elif cl == 'Female':                    col_rename[c] = 'Female_2026'
    elif 'Third' in cl:                     col_rename[c] = 'ThirdGender_2026'
elec2026.rename(columns=col_rename, inplace=True)
elec2026['Constituency'] = elec2026['Constituency'].str.strip()

# Fix 2 known spelling mismatches between ECI files
SPELLING_FIX = {'Palacodu': 'Palacode', 'Viluppuram': 'Villupuram'}
elec2026['Constituency'] = elec2026['Constituency'].replace(SPELLING_FIX)

# Check match quality before merging
unmatched = set(elec2026['Constituency']) - set(ac_df['Constituency'])
if unmatched:
    print(f'⚠️  {len(unmatched)} unmatched AC names in 2026 file: {unmatched}')
else:
    print('✅ All 234 AC names matched perfectly')

# Merge onto ac_df
ac_df = ac_df.merge(
    elec2026[['Constituency','Electors_2026','Male_2026','Female_2026']],
    on='Constituency', how='left'
)

# Estimate 2021 electors from turnout (TN avg turnout 2021 = 72.8%)
AVG_TURNOUT_2021 = 0.728
ac_df['Electors_2021_est'] = (ac_df['Total_Polled'] / AVG_TURNOUT_2021).round(0)
ac_df['Elector_Growth']    = ac_df['Electors_2026'] / ac_df['Electors_2021_est']
ac_df['New_Voters']        = (ac_df['Electors_2026'] - ac_df['Electors_2021_est']).clip(0)

# Sanity check
missing_2026 = ac_df['Electors_2026'].isna().sum()
print(f'✅ Cell 4 complete — elector data merged')
print(f'   ACs missing 2026 data : {missing_2026}  (should be 0)')
print(f'   Est. 2021 electors    : {ac_df["Electors_2021_est"].sum():,.0f}')
print(f'   2026 electors         : {ac_df["Electors_2026"].sum():,.0f}')
print(f'   Net new voters        : {ac_df["New_Voters"].sum():,.0f}')
print(f'   Mean growth per AC    : {ac_df["Elector_Growth"].mean():.3f}x')

In [ ]:

# CELL 5 — ELECTOR GROWTH VISUALISATION


fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].hist(ac_df['Elector_Growth'].dropna(), bins=30,
             color='steelblue', edgecolor='white')
axes[0].axvline(ac_df['Elector_Growth'].mean(), color='crimson', lw=2,
                linestyle='--', label=f"Mean: {ac_df['Elector_Growth'].mean():.3f}x")
axes[0].set_title('Elector Growth Rate per AC (2021→2026)', fontweight='bold')
axes[0].set_xlabel('Growth Multiplier'); axes[0].set_ylabel('Count')
axes[0].legend()

top10 = ac_df.nlargest(10, 'New_Voters')[['Constituency','New_Voters']]
axes[1].barh(top10['Constituency'], top10['New_Voters']/1000, color='teal')
axes[1].set_title('Top 10 ACs by New Voters Added (2026)', fontweight='bold')
axes[1].set_xlabel('New Voters (thousands)')

plt.tight_layout()
plt.savefig('elector_growth.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Cell 5 complete')

In [ ]:

# CELL 6 — 2021 ACTUAL VOTE-SHARE SCATTER (Baseline Visualisation)


fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Scatter: DMK% vs AIADMK%, coloured by winner
colors_map = ac_df['DMK_Won'].map({1: '#E63946', 0: '#457B9D'})
axes[0].scatter(ac_df['DMK_Alliance_Share']*100,
                ac_df['AIADMK_Alliance_Share']*100,
                c=colors_map, alpha=0.75, s=65, edgecolors='white', linewidth=0.4)
axes[0].axline((0,0), slope=1, color='black', linestyle='--', lw=1, alpha=0.4, label='Tie line')
red_p  = mpatches.Patch(color='#E63946', label='DMK Alliance wins')
blue_p = mpatches.Patch(color='#457B9D', label='AIADMK Alliance wins')
axes[0].legend(handles=[red_p, blue_p])
axes[0].set_xlabel('DMK Alliance Vote Share (%)')
axes[0].set_ylabel('AIADMK Alliance Vote Share (%)')
axes[0].set_title('2021 AC Outcomes — Alliance Vote Shares', fontweight='bold')

# NTK spoiler effect
ntk_bins = pd.cut(ac_df['NTK_share']*100,
                   bins=[0,5,8,12,16,30],
                   labels=['0–5%','5–8%','8–12%','12–16%','>16%'])
ntk_group = ac_df.groupby(ntk_bins, observed=False)['DMK_Won'].agg(['mean','count']).reset_index()
ntk_group.columns = ['NTK_Range','DMK_Win_Rate','Count']
bars = axes[1].bar(ntk_group['NTK_Range'].astype(str),
                   ntk_group['DMK_Win_Rate']*100,
                   color='#E63946', alpha=0.82, edgecolor='white')
for bar, row in zip(bars, ntk_group.itertuples()):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1.5,
                 f'n={row.Count}', ha='center', fontsize=9)
axes[1].set_ylim(0,100)
axes[1].set_title('DMK Alliance Win Rate vs NTK Vote Share (2021)', fontweight='bold')
axes[1].set_xlabel('NTK Vote Share Range')
axes[1].set_ylabel('DMK Alliance Win Rate (%)')

plt.tight_layout()
plt.savefig('vote_to_seat_2021.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Cell 6 complete')

In [ ]:

# CELL 7 — TVK SPLIT FUNCTION


def apply_tvk_split(
        ac_base: pd.DataFrame,
        tvk_state_share:    float = 0.15,
        drain_from_aiadmk:  float = 0.50,
        drain_from_dmk:     float = 0.25,
        drain_from_others:  float = 0.25,
        ac_variance:        float = 0.04,
        random_state:       int   = 42
) -> pd.DataFrame:
    """
    Simulate TVK entering 2026 race.
    TVK draws votes proportionally from AIADMK (50%), DMK (25%), Others (25%).
    Urban ACs (larger electorates) get higher TVK share.
    """
    rng = np.random.default_rng(random_state)
    df  = ac_base.copy()

    # Urban proxy: larger electorate → higher TVK share
    urban_factor = (df['Electors_2026'] / df['Electors_2026'].median()).clip(0.6, 1.5)
    tvk_raw = rng.normal(tvk_state_share * urban_factor, ac_variance, len(df)).clip(0.02, 0.35)
    df['TVK_share_raw'] = tvk_raw

    df['AIADMK_Alliance_Share_26'] = df['AIADMK_Alliance_Share'] - tvk_raw * drain_from_aiadmk
    df['DMK_Alliance_Share_26']    = df['DMK_Alliance_Share']    - tvk_raw * drain_from_dmk
    df['Others_share_26']          = df['Others_share']          - tvk_raw * drain_from_others

    for col in ['AIADMK_Alliance_Share_26','DMK_Alliance_Share_26','Others_share_26']:
        df[col] = df[col].clip(0.02)

    total = (df['DMK_Alliance_Share_26'] + df['AIADMK_Alliance_Share_26'] +
             df['TVK_share_raw']          + df['Others_share_26'])
    df['DMK_Alliance_Share_26']    /= total
    df['AIADMK_Alliance_Share_26'] /= total
    df['TVK_share_26']              = df['TVK_share_raw'] / total
    df['Others_share_26']          /= total

    return df


def scale_for_new_voters(
        df: pd.DataFrame,
        assumed_turnout_2026: float = 0.72,
        new_voter_prefs: dict = None
) -> pd.DataFrame:
    """
    Adjusts shares by routing new 2026 voters to parties based on survey prefs.
    Default: DMK 40%, TVK 25%, AIADMK 20%, Others 15%.
    Adds _adj columns and Winner_adj column.
    """
    if new_voter_prefs is None:
        new_voter_prefs = {
            'DMK_Alliance':    0.40,
            'AIADMK_Alliance': 0.20,
            'TVK':             0.25,
            'Others':          0.15,
        }

    d = df.copy()
    d['Est_New_Votes'] = (d['New_Voters'] * assumed_turnout_2026).clip(0)

    d['DMK_V_base']    = d['DMK_Alliance_Share_26']    * d['Total_Polled']
    d['AIADMK_V_base'] = d['AIADMK_Alliance_Share_26'] * d['Total_Polled']
    d['TVK_V_base']    = d['TVK_share_26']              * d['Total_Polled']
    d['Others_V_base'] = d['Others_share_26']           * d['Total_Polled']

    d['DMK_V_adj']    = d['DMK_V_base']    + d['Est_New_Votes'] * new_voter_prefs['DMK_Alliance']
    d['AIADMK_V_adj'] = d['AIADMK_V_base'] + d['Est_New_Votes'] * new_voter_prefs['AIADMK_Alliance']
    d['TVK_V_adj']    = d['TVK_V_base']    + d['Est_New_Votes'] * new_voter_prefs['TVK']
    d['Others_V_adj'] = d['Others_V_base'] + d['Est_New_Votes'] * new_voter_prefs['Others']

    total_adj = d['DMK_V_adj'] + d['AIADMK_V_adj'] + d['TVK_V_adj'] + d['Others_V_adj']
    d['DMK_share_adj']    = d['DMK_V_adj']    / total_adj
    d['AIADMK_share_adj'] = d['AIADMK_V_adj'] / total_adj
    d['TVK_share_adj']    = d['TVK_V_adj']    / total_adj
    d['Others_share_adj'] = d['Others_V_adj'] / total_adj

    share_cols = ['DMK_share_adj','AIADMK_share_adj','TVK_share_adj','Others_share_adj']
    label_map  = {
        'DMK_share_adj':    'DMK_Alliance',
        'AIADMK_share_adj': 'AIADMK_Alliance',
        'TVK_share_adj':    'TVK',
        'Others_share_adj': 'Others',
    }
    d['Winner_adj'] = d[share_cols].idxmax(axis=1).map(label_map)
    return d


def run_monte_carlo(
        ac_base: pd.DataFrame,
        n_sims: int = 10_000,
        uncertainty: float = 0.07,
        random_state: int = 0
) -> pd.DataFrame:
    """
    Run n_sims elections by adding Normal(0, uncertainty) noise per AC.
    Uses FPTP winner per AC.  Returns DataFrame of seat counts per sim.
    """
    share_cols = ['DMK_share_adj','AIADMK_share_adj','TVK_share_adj','Others_share_adj']
    label_map  = {
        0: 'DMK_Alliance',
        1: 'AIADMK_Alliance',
        2: 'TVK',
        3: 'Others',
    }
    rng    = np.random.default_rng(random_state)
    base   = ac_base[share_cols].values.copy()
    n_ac   = len(ac_base)
    results = []

    for _ in range(n_sims):
        noise  = rng.normal(0, uncertainty, (n_ac, 4))
        sim    = np.clip(base + noise, 0.01, None)
        sim    = sim / sim.sum(axis=1, keepdims=True)
        winner = sim.argmax(axis=1)
        cnt    = Counter(winner)
        results.append({
            'DMK_Alliance':    cnt.get(0, 0),
            'AIADMK_Alliance': cnt.get(1, 0),
            'TVK':             cnt.get(2, 0),
            'Others':          cnt.get(3, 0),
        })

    return pd.DataFrame(results)


print('✅ Cell 7 complete — TVK split, elector scaling, and Monte Carlo functions defined')

In [ ]:

# CELL 8 — BUILD 2026 BASE SCENARIO (TVK = 15%)


# Step A: Apply TVK split
ac_2026_tvk = apply_tvk_split(ac_df, tvk_state_share=0.15)

# Step B: Scale for new voters
ac_2026_adj = scale_for_new_voters(ac_2026_tvk)

print('📊 Statewide share estimates — Base Scenario (TVK = 15%):')
for col, label in [
    ('DMK_share_adj',    'DMK Alliance   '),
    ('AIADMK_share_adj', 'AIADMK Alliance'),
    ('TVK_share_adj',    'TVK            '),
    ('Others_share_adj', 'Others         '),
]:
    print(f'  {label}: {ac_2026_adj[col].mean()*100:.1f}%')

print()
print('📊 Point-estimate seat counts (before uncertainty):')
print(ac_2026_adj['Winner_adj'].value_counts().to_string())
print()
print('✅ Cell 8 complete — ac_2026_adj is ready')

In [ ]:

# CELL 9 — LOGISTIC REGRESSION MODEL


ML_FEATURES = ['DMK_Alliance_Share','AIADMK_Alliance_Share',
               'NTK_share','Others_share','Electors_2021_est']
TARGET = 'DMK_Won'

X = ac_df[ML_FEATURES].fillna(0)
y = ac_df[TARGET]

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('lr',     LogisticRegression(C=1.0, max_iter=2000, random_state=42))
])

skf    = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_acc = cross_val_score(pipe, X, y, cv=skf, scoring='accuracy')
cv_f1  = cross_val_score(pipe, X, y, cv=skf, scoring='f1')

pipe.fit(X, y)
y_pred = pipe.predict(X)

print('📈 Logistic Regression — trained on 2021 AC real data')
print(f'  Cross-val Accuracy (5-fold) : {cv_acc.mean():.3f} ± {cv_acc.std():.3f}')
print(f'  Cross-val F1      (5-fold) : {cv_f1.mean():.3f} ± {cv_f1.std():.3f}')
print()
print(classification_report(y, y_pred, target_names=['AIADMK Alliance','DMK Alliance']))

# Confusion matrix
cm = confusion_matrix(y, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred AIADMK','Pred DMK'],
            yticklabels=['Actual AIADMK','Actual DMK'])
plt.title('Confusion Matrix — 2021 AC Model', fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Coefficients
coef_df = pd.DataFrame({'Feature': ML_FEATURES,
                         'Coef':    pipe.named_steps['lr'].coef_[0]
                        }).sort_values('Coef')
colors_c = ['#E63946' if c < 0 else '#1a7f5e' for c in coef_df['Coef']]
plt.figure(figsize=(10,4))
plt.barh(coef_df['Feature'], coef_df['Coef'], color=colors_c, edgecolor='white')
plt.axvline(0, color='black', lw=0.8)
plt.title('LR Coefficients — DMK Win Drivers', fontweight='bold')
plt.tight_layout()
plt.savefig('lr_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()

# Attach win probability to 2026 df
ac_2026_adj['LR_DMK_WinProb'] = pipe.predict_proba(X)[:,1]

print('✅ Cell 9 complete')

In [ ]:

# CELL 10 — MONTE CARLO SIMULATION (10,000 runs)


print('🎲 Running 10,000-simulation Monte Carlo...')
mc = run_monte_carlo(ac_2026_adj, n_sims=10_000, uncertainty=0.07)
print('✅ Done!\n')

print('═'*62)
print('  TAMIL NADU 2026 — MONTE CARLO SEAT PROJECTION')
print('  Base: TVK=15% | elector-adjusted | ±7% per-AC noise')
print('═'*62)
print(f'  Total seats: 234     Majority threshold: 117')
print('─'*62)
print(f'  {"Bloc":<22} {"Mean":>6}  {"80% CI":>14}  {"P(≥117)":>10}')
print('─'*62)
for bloc in ['DMK_Alliance','AIADMK_Alliance','TVK','Others']:
    m      = mc[bloc].mean()
    lo, hi = np.percentile(mc[bloc], [10, 90])
    pm     = (mc[bloc] >= 117).mean()
    print(f'  {bloc:<22} {m:>6.1f}  {lo:>5.0f}–{hi:<6.0f}  {pm:>9.1%}')
print('─'*62)
hung = (mc.max(axis=1) < 117).mean()
print(f'  Hung Assembly probability : {hung:.1%}')
print('═'*62)

In [ ]:

# CELL 11 — MONTE CARLO DISTRIBUTION PLOTS


PARTY_COLORS = {
    'DMK_Alliance':    '#E63946',
    'AIADMK_Alliance': '#457B9D',
    'TVK':             '#2DC653',
    'Others':          '#F4A261',
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, bloc in zip(axes.flatten(), PARTY_COLORS):
    data   = mc[bloc]
    lo, hi = np.percentile(data, [10, 90])
    mean_v = data.mean()
    ax.hist(data, bins=40, color=PARTY_COLORS[bloc], alpha=0.85, edgecolor='white')
    ax.axvline(mean_v, color='black',  lw=2, ls='--', label=f'Mean: {mean_v:.0f}')
    ax.axvline(lo,     color='gray',   lw=1.5, ls=':')
    ax.axvline(hi,     color='gray',   lw=1.5, ls=':', label=f'80% CI: {lo:.0f}–{hi:.0f}')
    ax.axvline(117,    color='gold',   lw=2, alpha=0.9, label='Majority (117)')
    ax.set_title(f'{bloc.replace("_"," ")} — Seat Distribution', fontweight='bold')
    ax.set_xlabel('Seats'); ax.set_ylabel('# Simulations')
    ax.legend(fontsize=9)

plt.suptitle('Monte Carlo: Tamil Nadu 2026 Seat Projections\n'
             '(TVK=15%, n=10,000 simulations, ±7% per-AC noise)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('monte_carlo_2026.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Cell 11 complete')

In [ ]:

# CELL 12 — BACKTEST: Predict 2021 from 2016 baseline


rng16   = np.random.default_rng(2016)
n_ac    = len(ac_df)

# Simulate 2016 per-AC shares (AIADMK won 136 seats, ~44.8% share)
dmk16    = rng16.normal(0.316, 0.085, n_ac).clip(0.10, 0.60)
aiadmk16 = rng16.normal(0.448, 0.080, n_ac).clip(0.15, 0.70)
ntk16    = rng16.normal(0.055, 0.020, n_ac).clip(0.01, 0.15)
oth16    = np.clip(1 - dmk16 - aiadmk16 - ntk16, 0.02, 0.40)
total16  = dmk16 + aiadmk16 + ntk16 + oth16
dmk16 /= total16; aiadmk16 /= total16; ntk16 /= total16; oth16 /= total16

# Apply swing: anti-AIADMK incumbency (+6% DMK, -8% AIADMK)
ac_val = ac_df.copy()
ac_val['DMK_Alliance_Share']    = (dmk16    + 0.06).clip(0.05)
ac_val['AIADMK_Alliance_Share'] = (aiadmk16 - 0.08).clip(0.05)
ac_val['NTK_share']             = ntk16
ac_val['Others_share']          = oth16

# Re-normalise
total_sw = (ac_val['DMK_Alliance_Share'] + ac_val['AIADMK_Alliance_Share'] +
            ac_val['NTK_share']           + ac_val['Others_share'])
for col in ['DMK_Alliance_Share','AIADMK_Alliance_Share','NTK_share','Others_share']:
    ac_val[col] /= total_sw

# No TVK in 2021 — add trivial TVK share to pass into MC function
ac_val['DMK_Alliance_Share_26']    = ac_val['DMK_Alliance_Share']
ac_val['AIADMK_Alliance_Share_26'] = ac_val['AIADMK_Alliance_Share']
ac_val['TVK_share_26']             = 0.001
ac_val['Others_share_26']          = ac_val['Others_share']
ac_val = scale_for_new_voters(ac_val)

mc_val = run_monte_carlo(ac_val, n_sims=5000)

ACTUAL_2021 = {'DMK_Alliance': 159, 'AIADMK_Alliance': 75, 'TVK': 0, 'Others': 0}

print('━'*66)
print('  BACKTEST: Predicted 2021 (from 2016 base + swing) vs Actual')
print('━'*66)
print(f'  {"Bloc":<22} {"Predicted":>10}  {"80% CI":>14}  {"Actual":>8}  In CI?')
print('─'*66)
for bloc in ['DMK_Alliance','AIADMK_Alliance']:
    pred_m  = mc_val[bloc].mean()
    lo, hi  = np.percentile(mc_val[bloc], [10,90])
    actual  = ACTUAL_2021[bloc]
    in_ci   = '✅' if lo <= actual <= hi else '⚠️  outside CI'
    print(f'  {bloc:<22} {pred_m:>10.1f}  {lo:>5.0f}–{hi:<6.0f}  {actual:>8}  {in_ci}')
print('━'*66)
print('✅ Cell 12 complete')

In [ ]:

# CELL 13 — TVK SENSITIVITY SWEEP (8% to 22%)


TVK_SCENARIOS = [0.08, 0.10, 0.12, 0.15, 0.18, 0.20, 0.22]
scenario_rows = []

for tvk in TVK_SCENARIOS:
    ac_s = apply_tvk_split(ac_df, tvk_state_share=tvk, random_state=int(tvk*1000))
    ac_s = scale_for_new_voters(ac_s)
    mc_s = run_monte_carlo(ac_s, n_sims=3000)
    lo_d, hi_d = np.percentile(mc_s['DMK_Alliance'],    [10,90])
    scenario_rows.append({
        'TVK_pct':           f'{int(tvk*100)}%',
        'DMK_mean':          mc_s['DMK_Alliance'].mean(),
        'AIADMK_mean':       mc_s['AIADMK_Alliance'].mean(),
        'TVK_mean':          mc_s['TVK'].mean(),
        'Others_mean':       mc_s['Others'].mean(),
        'DMK_lo': lo_d, 'DMK_hi': hi_d,
        'DMK_majority_prob':    (mc_s['DMK_Alliance']    >= 117).mean(),
        'AIADMK_majority_prob': (mc_s['AIADMK_Alliance'] >= 117).mean(),
        'TVK_majority_prob':    (mc_s['TVK']              >= 117).mean(),
        'Hung_prob':            (mc_s.max(axis=1)         <  117).mean(),
    })
    print(f"  TVK {int(tvk*100):>3}% → DMK:{mc_s['DMK_Alliance'].mean():.0f}  "
          f"AIADMK:{mc_s['AIADMK_Alliance'].mean():.0f}  TVK:{mc_s['TVK'].mean():.0f}")

sens_df = pd.DataFrame(scenario_rows)

# ── Plot seats ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
x  = np.arange(len(TVK_SCENARIOS))
w  = 0.26
xl = [f'{int(t*100)}%' for t in TVK_SCENARIOS]

ax = axes[0]
b1 = ax.bar(x-w,  sens_df['DMK_mean'],    w, label='DMK Alliance',    color='#E63946', alpha=0.88)
b2 = ax.bar(x,    sens_df['AIADMK_mean'], w, label='AIADMK Alliance', color='#457B9D', alpha=0.88)
b3 = ax.bar(x+w,  sens_df['TVK_mean'],    w, label='TVK',             color='#2DC653', alpha=0.88)
ax.axhline(117, color='gold', lw=2, ls='--', label='Majority (117)')
ax.set_xticks(x); ax.set_xticklabels(xl)
ax.set_xlabel('TVK Vote Share'); ax.set_ylabel('Seats')
ax.set_title('Seat Projection vs TVK Vote Share', fontweight='bold')
ax.legend(); ax.set_ylim(0, 220)
for bars in [b1, b2, b3]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2, h+2, f'{h:.0f}',
                ha='center', fontsize=8, fontweight='bold')

# ── Probability lines ──────────────────────────────────────────────────────────
ax2 = axes[1]
ax2.plot(sens_df['TVK_pct'], sens_df['DMK_majority_prob']*100,
         'o-', color='#E63946', lw=2.5, ms=8, label='DMK majority prob')
ax2.plot(sens_df['TVK_pct'], sens_df['Hung_prob']*100,
         's--', color='gray', lw=2, ms=7, label='Hung Assembly prob')
ax2.plot(sens_df['TVK_pct'], sens_df['TVK_majority_prob']*100,
         '^:', color='#2DC653', lw=2, ms=7, label='TVK majority prob')
ax2.set_xlabel('TVK Vote Share')
ax2.set_ylabel('Probability (%)')
ax2.set_title('Majority & Hung Assembly Probabilities', fontweight='bold')
ax2.legend(); ax2.set_ylim(0, 100)

plt.suptitle('Tamil Nadu 2026 — TVK Sensitivity Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('tvk_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n', sens_df[['TVK_pct','DMK_mean','AIADMK_mean','TVK_mean',
                      'DMK_majority_prob','Hung_prob']].to_string(index=False))
print('✅ Cell 13 complete')

In [ ]:

# CELL 14 — MARGINAL SWING SEATS (Top 20 closest to 50% win prob)


ac_2026_adj['Seat_Risk'] = 'Safe'
ac_2026_adj.loc[
    (ac_2026_adj['LR_DMK_WinProb'] > 0.40) &
    (ac_2026_adj['LR_DMK_WinProb'] < 0.60), 'Seat_Risk'] = 'Marginal'
ac_2026_adj.loc[ac_2026_adj['LR_DMK_WinProb'] >= 0.60, 'Seat_Risk'] = 'DMK Safe'
ac_2026_adj.loc[ac_2026_adj['LR_DMK_WinProb'] <= 0.40, 'Seat_Risk'] = 'AIADMK Safe'

print('Seat Risk Classification:')
print(ac_2026_adj['Seat_Risk'].value_counts().to_string())
print()

marginal = (
    ac_2026_adj
    .assign(Dist_from_50=(ac_2026_adj['LR_DMK_WinProb'] - 0.5).abs())
    .nsmallest(20, 'Dist_from_50')
    [['Constituency','Winner_Party','LR_DMK_WinProb',
      'DMK_share_adj','AIADMK_share_adj','TVK_share_adj','Winner_adj']]
    .copy()
)

fig, ax = plt.subplots(figsize=(12, 8))
colors_m = ['#E63946' if p > 0.5 else '#457B9D' for p in marginal['LR_DMK_WinProb']]
ax.barh(marginal['Constituency'],
        marginal['LR_DMK_WinProb']*100 - 50,
        color=colors_m, alpha=0.82, left=50, edgecolor='white')
ax.axvline(50, color='black', lw=1.5, ls='--')
ax.set_xlabel('DMK Alliance Win Probability (%)')
ax.set_title('Top 20 Swing Seats — Tamil Nadu 2026\n'
             '(Constituencies closest to 50% win probability)', fontweight='bold')
ax.set_xlim(35, 65)
plt.tight_layout()
plt.savefig('marginal_seats.png', dpi=150, bbox_inches='tight')
plt.show()

display(marginal[['Constituency','Winner_Party','LR_DMK_WinProb','Winner_adj']]
        .rename(columns={'Winner_Party':'2021 Winner',
                         'LR_DMK_WinProb':'DMK Win Prob 2026',
                         'Winner_adj':'Projected 2026'})
        .round(3))
print('✅ Cell 14 complete')

In [ ]:

# CELL 15 — FINAL PROJECTION CHART


means = mc.mean()
blocs = ['DMK_Alliance','AIADMK_Alliance','TVK','Others']
labels_full = ['DMK Alliance','AIADMK Alliance','TVK','Others']
colors_pie  = ['#E63946','#457B9D','#2DC653','#F4A261']

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Pie
wedges, texts, autotexts = axes[0].pie(
    [means[b] for b in blocs],
    labels=labels_full, colors=colors_pie,
    autopct='%1.1f%%', startangle=90,
    wedgeprops={'edgecolor':'white','linewidth':2}
)
for t in autotexts: t.set_fontsize(12); t.set_fontweight('bold')
axes[0].set_title('Tamil Nadu 2026 — Projected Seat Share\n(Central: TVK=15%)', fontweight='bold')
seat_labels = [f'{l}: {int(means[b])} seats' for l,b in zip(labels_full,blocs)]
axes[0].legend(seat_labels, loc='lower center', bbox_to_anchor=(0.5,-0.12), ncol=2)

# Bar with CI
means_ = [mc[b].mean() for b in blocs]
err_lo = [m - np.percentile(mc[b],10) for m,b in zip(means_,blocs)]
err_hi = [np.percentile(mc[b],90) - m for m,b in zip(means_,blocs)]

bars = axes[1].bar(labels_full, means_, color=colors_pie, alpha=0.88, edgecolor='white')
axes[1].errorbar(labels_full, means_, yerr=[err_lo, err_hi],
                 fmt='none', color='black', capsize=8, lw=2)
axes[1].axhline(117, color='gold', lw=2.5, ls='--', label='Majority (117)')
axes[1].set_ylabel('Projected Seats')
axes[1].set_title('Seats with 80% Confidence Interval', fontweight='bold')
axes[1].legend()
for bar, m in zip(bars, means_):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
                 f'{m:.0f}', ha='center', fontweight='bold', fontsize=13)

plt.suptitle('Tamil Nadu 2026 Assembly Election — Final Projection\n'
             'Source: Real 2021 ECI Data + 2026 Elector Rolls | n=10,000 simulations',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('final_projection_2026.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Cell 15 complete')

In [ ]:

# CELL 16 — EXPORT FULL AC-LEVEL PROJECTION TABLE


export_cols = [
    'Constituency',
    'Winner_Party',        # 2021 winner
    'Winner_Alliance',     # 2021 alliance
    'Electors_2026',       # from your 2026 file
    'New_Voters',          # growth since 2021
    'Elector_Growth',      # multiplier
    'DMK_Alliance_Share',  # 2021 actual
    'AIADMK_Alliance_Share',
    'NTK_share',
    'DMK_share_adj',       # 2026 projected
    'AIADMK_share_adj',
    'TVK_share_adj',
    'Others_share_adj',
    'Winner_adj',          # 2026 projected winner
    'LR_DMK_WinProb',      # model probability
    'Seat_Risk',           # Safe / Marginal
]

out = ac_2026_adj[[c for c in export_cols if c in ac_2026_adj.columns]].copy()

# Save in same folder as notebook
OUT_PATH = r"C:\Users\SamuelJoshuaRaj\OneDrive - CYGNUSA Technologies\Desktop\TN_2026_AC_Projections.csv"
out.to_csv(OUT_PATH, index=False)
print(f'✅ Exported {len(out)} rows → {OUT_PATH}')
print()
display(out.head(10))

In [ ]:

# CELL 17 — LIVE POLL TRACKER  (add real surveys here as they release)


# ── ADD REAL SURVEY DATA BELOW AS IT RELEASES ─────────────────────────────────
poll_tracker = pd.DataFrame([
    {'Date':'2025-09-01','Agency':'Survey India (placeholder)',
     'DMK_Alliance':38,'AIADMK_Alliance':30,'TVK':13,'Others':19},
    {'Date':'2025-11-15','Agency':'C-Voter (placeholder)',
     'DMK_Alliance':36,'AIADMK_Alliance':28,'TVK':16,'Others':20},
    {'Date':'2026-01-20','Agency':'Lokniti-CSDS (placeholder)',
     'DMK_Alliance':35,'AIADMK_Alliance':27,'TVK':18,'Others':20},
])
poll_tracker['Date'] = pd.to_datetime(poll_tracker['Date'])

fig, ax = plt.subplots(figsize=(12, 6))
plot_info = [
    ('DMK_Alliance',    '#E63946', 'o'),
    ('AIADMK_Alliance', '#457B9D', 's'),
    ('TVK',             '#2DC653', '^'),
]
for col, color, marker in plot_info:
    ax.plot(poll_tracker['Date'], poll_tracker[col],
            marker=marker, color=color, lw=2.5, ms=10,
            label=col.replace('_',' '))
    ax.annotate(f"{poll_tracker[col].iloc[-1]}%",
                (poll_tracker['Date'].iloc[-1], poll_tracker[col].iloc[-1]),
                xytext=(8,0), textcoords='offset points',
                color=color, fontweight='bold', fontsize=11)

ax.set_ylabel('Vote Share (%)', fontsize=12)
ax.set_xlabel('Survey Date')
ax.set_title('Pre-Poll Vote Share Tracker — Tamil Nadu 2026\n'
             '(Replace placeholder rows with real survey data)', fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, 55)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('poll_tracker.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Cell 17 complete — all done!')